# Métodos das camadas Bronze, Silver e Gold

Este notebook apresenta os principais métodos usados no pipeline e executa somente operações demonstrativas ou de leitura.

## Fluxo das camadas

A Bronze recebe os dados originais, a Silver limpa e valida os dados, e a Gold combina as informações em tabelas prontas para análise, machine learning e decisão.

In [ ]:
from inspect import signature
from pathlib import Path
import polars as pl

projeto = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
dados_gold = projeto / 'data' / 'gold'

## Bronze

A Bronze preserva os dados recebidos das fontes externas. Ela registra metadados, evita duplicatas por hash e envia registros inválidos para quarentena.

In [ ]:
from credito_pa.bronze.estban_file import month_range, pending_competencias, parse_estban_csv
from credito_pa.bronze.ibge_api import chunk, sidra_url, parse_sidra_rows, flatten_municipio
from credito_pa.bronze.inpe_file import parse_inpe_csv
from credito_pa.bronze.scr_file import detect_encoding, scan_uf

metodos_bronze = {
    'ESTBAN': [month_range, pending_competencias, parse_estban_csv],
    'IBGE': [chunk, sidra_url, parse_sidra_rows, flatten_municipio],
    'INPE': [parse_inpe_csv],
    'SCR': [detect_encoding, scan_uf],
}

{fonte: [f'{metodo.__name__}{signature(metodo)}' for metodo in metodos] for fonte, metodos in metodos_bronze.items()}

### Métodos principais da Bronze

`month_range` cria competências mensais. `pending_competencias` identifica meses ainda não carregados. `parse_estban_csv`, `parse_inpe_csv` e `parse_sidra_rows` transformam respostas brutas em registros. `chunk` divide listas grandes em lotes. `detect_encoding` identifica a codificação de arquivos. `scan_uf` filtra dados do SCR para o Pará. As funções `ingest_*` coordenam download, escrita e controle da ingestão.

## Silver

A Silver transforma os dados brutos em dados tipados, normalizados e validados. Registros que não passam pelas regras são enviados para quarentena.

In [ ]:
from credito_pa.silver.base import br_number, plain_number, int_number, latest_by_key, normalize_name, month_end_from_yyyymm

dados_numericos = pl.DataFrame({
    'valor_brasileiro': ['1.234,56', '-', '98,10'],
    'valor_simples': ['12.5', '7', 'erro'],
    'quantidade': ['10', '2', 'x']
})

dados_tipados = dados_numericos.with_columns([
    br_number('valor_brasileiro'),
    plain_number('valor_simples'),
    int_number('quantidade')
])
dados_tipados

In [ ]:
dados_nomes = pl.DataFrame({'nome_original': ['São Félix do Xingu', '  Belém  ', None]})
dados_nomes.with_columns(pl.col('nome_original').map_elements(normalize_name, return_dtype=pl.Utf8).alias('nome_normalizado'))

In [ ]:
dados_competencias = pl.DataFrame({'competencia': ['202401', '202402', '202412']})
dados_competencias.with_columns(month_end_from_yyyymm('competencia').alias('data_competencia'))

### Métodos principais da Silver

`br_number`, `plain_number` e `int_number` convertem textos em números. `normalize_name` padroniza nomes. `month_end_from_yyyymm` converte uma competência como `202401` para o último dia do mês. `latest_by_key` remove duplicatas mantendo o registro mais recente. `read_bronze` lê os dados da Bronze. `finalize` aplica o contrato, separa inválidos, verifica a chave primária e salva a tabela Silver.

## Gold

A Gold reúne os dados tratados em produtos analíticos. As tabelas Gold são validadas antes de serem gravadas.

In [ ]:
from credito_pa.gold.base import parse_month, month_grid, read_gold
from credito_pa.gold.fato_mensal import ano_ibge_referencia

mes_inicial = parse_month('2023-01')
mes_final = parse_month('2023-04')
meses = month_grid('2023-01', '2023-04')

mes_inicial, mes_final, meses

In [ ]:
datas_de_competencia = pl.DataFrame({
    'data_competencia': [pl.date(2023, 6, 30), pl.date(2023, 12, 31), pl.date(2024, 12, 31)]
})
datas_de_competencia.with_columns(
    ano_ibge_referencia(pl.col('data_competencia'), defasagem_anos=2, mes_publicacao=12).alias('ano_ibge_conhecido')
)

In [ ]:
tabelas_gold_disponiveis = sorted(p.stem for p in dados_gold.glob('*.parquet'))
tabelas_gold_disponiveis

In [ ]:
nome_tabela_fato = 'fato_credito_municipio_mes'
caminho_tabela_fato = dados_gold / f'{nome_tabela_fato}.parquet'
fato_credito_municipio_mes = pl.read_parquet(caminho_tabela_fato) if caminho_tabela_fato.exists() else pl.DataFrame()
fato_credito_municipio_mes.head()

### Métodos principais da Gold

`parse_month` e `month_grid` trabalham com competências mensais. `ano_ibge_referencia` escolhe o ano do PIB que já estava disponível na data. `build_contexto_scr` agrega os indicadores do SCR por mês. `build_fato` combina crédito, IBGE, INPE e SCR por município e mês. `finalize_gold` valida o contrato, verifica a chave primária e grava o Parquet. `build_ranking` usa o modelo para calcular prioridades e `build_decisao` transforma os resultados em recomendações.

## Resultado

Bronze conserva a origem, Silver garante qualidade e Gold entrega tabelas prontas para análise e decisão.